# День 10: больше данных — обучение ещё и на валидации (без valid_sub) + подбор весов смеси (~2 ч)
Итог day9: регуляризация дообучению бейзлайна **не помогла** (ft_reg 0.66476 против 0.66679 у ft_swa); reg128b с более сильной регуляризацией — как reg128 на маске (0.66552) и лучшая из всех на всех строках (0.51175). Лучшей осталась отправленная тройка (0.67036).

В valid.parquet 1 873 последовательности с ответами, а для проверки мы используем только 600 (`valid_sub`). Остальные **1 273** можно добавить к обучению (+12% данных). `valid_sub` в обучение не попадает, так что сравнение моделей остаётся честным по последовательностям. Оговорка: если последовательности валидации взяты из тех же дней, что и `valid_sub`, их скор на `valid_sub` может быть чуть завышен — окончательно покажет лидерборд.

Запуски:
1. **ft2** — продолжение дообучения ft_swa (с его усреднёнными весами) на train + остаток валидации: 3 эпохи, lr 1e-4, без регуляризации (~35 мин).
2. **reg128m** — GRU-128 с нуля на train + остаток валидации, регуляризация как у reg128b, сид 2 (~1.3 ч).
3. Оценка, **подбор весов** для лучших смесей (раньше веса всегда были равными, а подбор каждый раз давал ft_swa больше веса) и сборка — если лучше отправленной тройки хотя бы на 0.0003.

Перед запуском: **T4 GPU**; загрузите на Drive вместо `wunder/day2` папку `хакатон/day2` — изменился `train.py`.

## 1. GPU + Drive + код + numba как у скорера

In [1]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install "numba==0.67.0" onnx 2>&1 | grep -v "dependency resolver\|requires numba\|incompatible" || true
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!grep -q "extra-parquet" train.py && echo "code: OK (training on extra sequences supported)" || echo "OLD CODE on Drive - upload the new day2 folder"

Tesla T4
Mounted at /content/drive
/content/day2
code: OK (training on extra sequences supported)


## 2. Данные: train.parquet и valid.parquet из стартового пакета (~6 мин)

In [2]:
import glob
URL = 'https://files.wundernn.io/wnn_connectome_starterpack.tar.gz'
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y aria2 > /dev/null 2>&1
!aria2c -c -x 16 -s 16 -k 1M --file-allocation=none --summary-interval=30 --console-log-level=warn -d /content -o pack.tar.gz $URL
!mkdir -p /content/sp && tar -xzf /content/pack.tar.gz -C /content/sp --wildcards '*datasets/train.parquet' '*datasets/valid.parquet' 2>&1 | grep -v LIBARCHIVE
!rm -f /content/pack.tar.gz
TRAIN_PQ = glob.glob('/content/sp/**/datasets/train.parquet', recursive=True)[0]
VALID_PQ = glob.glob('/content/sp/**/datasets/valid.parquet', recursive=True)[0]
HELD = W + '/colab_data/valid_sub.parquet'          # these 600 sequences stay out of training
EXTRA = f'--extra-parquet {VALID_PQ} --exclude-seqs-from {HELD}'
print(TRAIN_PQ, VALID_PQ, sep='\n')

 *** Download Progress Summary as of Fri Nov 13 13:53:51 2026 *** 
=
[#e9de08 5.9GiB/31GiB(19%) CN:16 DL:234MiB ETA:1m50s]
FILE: /content/pack.tar.gz
-

 *** Download Progress Summary as of Fri Nov 13 13:54:21 2026 *** 
=
[#e9de08 12GiB/31GiB(38%) CN:16 DL:189MiB ETA:1m43s]
FILE: /content/pack.tar.gz
-

 *** Download Progress Summary as of Fri Nov 13 13:54:51 2026 *** 
=
[#e9de08 17GiB/31GiB(57%) CN:16 DL:198MiB ETA:1m9s]
FILE: /content/pack.tar.gz
-

 *** Download Progress Summary as of Fri Nov 13 13:55:22 2026 *** 
=
[#e9de08 24GiB/31GiB(78%) CN:16 DL:229MiB ETA:29s]
FILE: /content/pack.tar.gz
-


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
e9de08|OK  |   213MiB/s|/content/pack.tar.gz

Status Legend:
(OK):download completed.
/content/sp/wnn_connectome_starterpack/datasets/train.parquet
/content/sp/wnn_connectome_starterpack/datasets/valid.parquet


## 3. Валидация — те же 600 последовательностей `valid_sub` (~1.5 мин)

In [3]:
!python prepare_data.py --src $HELD --out /content/data/valid

100/600 seqs  14s  eta 70s
200/600 seqs  21s  eta 43s
300/600 seqs  33s  eta 33s
400/600 seqs  42s  eta 21s
500/600 seqs  58s  eta 12s
600/600 seqs  71s  eta 0s
{
 "src": "/content/drive/MyDrive/wunder/colab_data/valid_sub.parquet",
 "n_seqs": 600,
 "seq_len": 20000,
 "n_features": 112,
 "dtype": "float32",
 "has_is_scored": true,
 "nan_features": 0,
 "inf_features": 0,
 "nonfinite_targets": 0,
 "float16_overflow": 0,
 "scored_rows": 1144931
}


## 4. ft2: продолжение дообучения ft_swa на train + остаток валидации (~35 мин)
В начале лога должно быть `train seqs 11880 ... + 1273 of 1873 sequences ... (without the 600 that are in valid_sub.parquet)`, а строка `VALID (start ...)` — 0.66679 (это ft_swa).

In [4]:
FT_SWA = '+'.join(W + f'/runs/ft_baseline_full/epoch_{k}.pt' for k in range(2, 6))
!python train.py --train-parquet $TRAIN_PQ $EXTRA --valid-dir /content/data/valid --out $W/runs/ft2 \
    --arch baseline --init-from $FT_SWA --keep-epochs --epochs 3 --batch 32 --chunk 1000 --lr 1e-4 --mse-weight 0.1 \
    --loader-workers 4

device=cuda amp=True Tesla T4
train seqs 11880 from /content/sp/wnn_connectome_starterpack/datasets/train.parquet + 1273 of 1873 sequences of /content/sp/wnn_connectome_starterpack/datasets/valid.parquet (without the 600 that are in valid_sub.parquet) (parquet, read on the fly by 4 threads); valid seqs 600
initialised from /content/drive/MyDrive/wunder/runs/ft_baseline_full/epoch_2.pt+/content/drive/MyDrive/wunder/runs/ft_baseline_full/epoch_3.pt+/content/drive/MyDrive/wunder/runs/ft_baseline_full/epoch_4.pt+/content/drive/MyDrive/wunder/runs/ft_baseline_full/epoch_5.pt
VALID (start = initial weights) {"seqs": 600, "pred_std": [0.5619000196456909, 0.5795000195503235], "WP_mask_std0.15": 0.66679, "WP_mask_std0.25": 0.66679, "WP_mask_std0.35": 0.66679, "WP_mask_std0.5": 0.66679, "WP_mask_std0.7": 0.66681, "WP_mask_std1.0": 0.66305, "WP_mask_std1.5": 0.64949, "WP_mask_raw": 0.66681, "WP_mask_best": 0.66681, "best_target_std": 0.7, "output_scale": [1.2457799911499023, 1.2079700231552124], 

## 5. reg128m: GRU-128 с нуля на train + остаток валидации (~1.3 ч)

In [5]:
!python train.py --train-parquet $TRAIN_PQ $EXTRA --valid-dir /content/data/valid --out $W/runs/reg128m \
    --arch gru --hidden 128 --epochs 8 --batch 64 --chunk 1000 --lr 2e-3 --keep-epochs --seed 2 \
    --dropout 0.3 --input-noise 0.1 --wd 2e-2 --loader-workers 4

device=cuda amp=True Tesla T4
train seqs 11880 from /content/sp/wnn_connectome_starterpack/datasets/train.parquet + 1273 of 1873 sequences of /content/sp/wnn_connectome_starterpack/datasets/valid.parquet (without the 600 that are in valid_sub.parquet) (parquet, read on the fly by 4 threads); valid seqs 600
params: 221,986
ep 0 batch 10/186 loss 0.8826 lr 1.34e-03 0.40M rows/s data-wait 39% eta 9.5 min
ep 0 batch 20/186 loss 0.7871 lr 2.00e-03 0.41M rows/s data-wait 39% eta 8.5 min
ep 0 batch 30/186 loss 0.7793 lr 2.00e-03 0.42M rows/s data-wait 39% eta 7.9 min
ep 0 batch 40/186 loss 0.7557 lr 2.00e-03 0.43M rows/s data-wait 40% eta 7.3 min
ep 0 batch 50/186 loss 0.7477 lr 1.99e-03 0.43M rows/s data-wait 40% eta 6.8 min
ep 0 batch 60/186 loss 0.7382 lr 1.99e-03 0.43M rows/s data-wait 40% eta 6.3 min
ep 0 batch 70/186 loss 0.7299 lr 1.99e-03 0.43M rows/s data-wait 40% eta 5.8 min
ep 0 batch 80/186 loss 0.7180 lr 1.99e-03 0.43M rows/s data-wait 40% eta 5.3 min
ep 0 batch 90/186 loss 0.722

## 6. Все модели на одной валидации (~12 мин)

In [ ]:
import os
def swa(run, ks):
    return '+'.join(f'{run}/epoch_{k}.pt' for k in ks)
CANDS = {
    'ft_swa':   FT_SWA,
    'ft2_swa':  swa(W + '/runs/ft2', range(0, 3)),
    'ft2_last': W + '/runs/ft2/epoch_2.pt',
    'gru_full': W + '/runs/gru_full/best.pt',
    'reg128':   W + '/runs/reg128/best.pt',
    'reg128b':  W + '/runs/reg128b/best.pt',
    'reg128m':  W + '/runs/reg128m/best.pt',
}
FOUND = {k: v for k, v in CANDS.items() if all(os.path.exists(p) for p in v.split('+'))}
print('models:', list(FOUND), '| missing:', [k for k in CANDS if k not in FOUND])
MODELS = ' '.join(f'--model {k}={v}' for k, v in FOUND.items())
!python ensemble_eval.py --valid-dir /content/data/valid --out /content/preds10 $MODELS --report $W/day10.json

models: ['ft_swa', 'ft2_swa', 'ft2_last', 'gru_full', 'reg128', 'reg128b', 'reg128m'] | missing: []
ft_swa         predicted (baseline, epoch 2, SWA of 4)
ft2_swa        predicted (baseline, epoch 0, SWA of 3)
ft2_last       predicted (baseline, epoch 2)
gru_full       predicted (gru, epoch 5)
reg128         predicted (gru, epoch 6)
reg128b        predicted (gru, epoch 6)
reg128m        predicted (gru, epoch 6)

validation: 600 sequences, 1,144,931 scored rows. WP on the mask (and on all required rows):
  ft_swa         0.66679   (0.51096)
  ft2_swa        0.66737   (0.51227)
  ft2_last       0.66699   (0.51144)
  gru_full       0.66370   (0.50515)
  reg128         0.66544   (0.50999)
  reg128b        0.66552   (0.51174)
  reg128m        0.66718   (0.51545)

correlation of predictions on scored rows (lower = more to gain from blending):
                     ft_swa    ft2_swa   ft2_last   gru_full     reg128    reg128b    reg128m
  ft_swa              1.000      0.999      0.999      0.

## 7. Подбор весов и выбор сабмита
Для трёх лучших смесей (до трёх моделей — по времени это 0.53x или 0.8x) перебираются веса с шагом 0.1; модели с нулевым весом выкидываются. Сборка — если лучший вариант лучше отправленной тройки (0.67036) хотя бы на 0.0003.

In [ ]:
import json, sys
import numpy as np
sys.path.insert(0, '/content/day2')
from ensemble_eval import standardize, blend, wp, weight_grid
from common import SEQ_LEN, WARMUP
eb = json.load(open(W + '/day10.json'))['equal_blends']
SENT = 'ft_swa+gru_full+reg128'
Y = np.load('/content/data/valid/targets.npy')
M = np.load('/content/data/valid/is_scored.npy') & (np.arange(SEQ_LEN) >= WARMUP)
Ym = Y[M]
Z = {n: standardize(np.load(f'/content/preds10/{n}.npy')[M]) for n in FOUND}
top = sorted(((v, k) for k, v in eb.items() if k.count('+') <= 2), reverse=True)[:3]
if SENT not in [k for _, k in top]:
    top.append((eb[SENT], SENT))
results = []
for v, k in top:
    names = k.split('+')
    grid = list(weight_grid(len(names), 0.1)) + [np.ones(len(names)) / len(names)]   # + exactly equal weights
    s, w = max((wp(Ym, blend([Z[n] for n in names], w)), tuple(float(x) for x in np.round(w, 3))) for w in grid)
    results.append((s, k, w))
    print(f'{k:34s} equal weights {v:.5f} | best weights {w} -> {s:.5f} ({s - eb[SENT]:+.5f} vs sent)')
BEST_S, BEST, BEST_W = max(results)
BUILD = BEST_S > eb[SENT] + 0.0003
PARTS = [(n, w) for n, w in zip(BEST.split('+'), BEST_W) if w > 0]
OUT = W + '/submissions/ens10_' + '_'.join(n for n, _ in PARTS)
SPEC = ' '.join(f'--model "{n}={FOUND[n]}@{w:.2f}"' for n, w in PARTS)
REF = W + '/submissions/gru128x2_v1'
print('\nbuild:', ' + '.join(f'{w:.2f}*{n}' for n, w in PARTS) if BUILD else 'nothing - no candidate beats the sent ensemble by 0.0003')

## 8. Сборка, проверка, скорость, скачивание (~10 мин)

In [ ]:
if globals().get('BUILD'):
    !python build_ensemble.py --valid-dir /content/data/valid --out $OUT $SPEC
    !python check_submission.py {OUT}.zip --data $HELD --seqs 3 --rows 20000 --ref-dir $REF
    !python bench_speed.py $REF $OUT --rows 5000 --repeats 5
    from google.colab import files
    files.download(OUT + '.zip')
else:
    print('nothing to build (or cell 7 was not run in this session)')

## Что прислать
Строки `VALID` из обоих обучений и вывод ячеек 6–8. Если zip собрался и проверка прошла — можно отправлять.